In [21]:
psg_path = "/home/maelic/Documents/SGG-Benchmark/datasets/psg/psg_train_val.json"

import json
from tqdm import tqdm

with open(psg_path, 'r') as f:
    psg = json.load(f)

print(psg.keys())
print(psg['data'][0])


dict_keys(['data', 'thing_classes', 'stuff_classes', 'predicate_classes', 'test_image_ids'])
{'file_name': 'train2017/000000417720.jpg', 'height': 640, 'width': 480, 'image_id': '107902', 'pan_seg_file_name': 'panoptic_train2017/000000417720.png', 'segments_info': [{'id': 3648962, 'category_id': 0, 'iscrowd': 0, 'isthing': 1, 'area': 42555}, {'id': 3293004, 'category_id': 0, 'iscrowd': 0, 'isthing': 1, 'area': 45880}, {'id': 4470325, 'category_id': 25, 'iscrowd': 0, 'isthing': 1, 'area': 22774}, {'id': 4545879, 'category_id': 116, 'iscrowd': 0, 'isthing': 0, 'area': 48824}, {'id': 7633023, 'category_id': 117, 'iscrowd': 0, 'isthing': 0, 'area': 44811}, {'id': 14079701, 'category_id': 119, 'iscrowd': 0, 'isthing': 0, 'area': 3417}, {'id': 3504233, 'category_id': 125, 'iscrowd': 0, 'isthing': 0, 'area': 97238}], 'relations': [[0, 1, 2], [0, 6, 14], [1, 2, 21], [1, 6, 14], [4, 3, 1], [5, 3, 0]], 'annotations': [{'bbox': [49.0, 162.0, 207.0, 617.0], 'bbox_mode': 0, 'category_id': 0}, {'bbo

In [16]:
print(psg['data'][0]['annotations'][0])
print(psg['data'][0]['relations'][0])
print(psg['data'][0])


{'bbox': [49.0, 162.0, 207.0, 617.0], 'bbox_mode': 0, 'category_id': 0}
[0, 1, 2]
{'file_name': 'train2017/000000417720.jpg', 'height': 640, 'width': 480, 'image_id': '107902', 'pan_seg_file_name': 'panoptic_train2017/000000417720.png', 'segments_info': [{'id': 3648962, 'category_id': 0, 'iscrowd': 0, 'isthing': 1, 'area': 42555}, {'id': 3293004, 'category_id': 0, 'iscrowd': 0, 'isthing': 1, 'area': 45880}, {'id': 4470325, 'category_id': 25, 'iscrowd': 0, 'isthing': 1, 'area': 22774}, {'id': 4545879, 'category_id': 116, 'iscrowd': 0, 'isthing': 0, 'area': 48824}, {'id': 7633023, 'category_id': 117, 'iscrowd': 0, 'isthing': 0, 'area': 44811}, {'id': 14079701, 'category_id': 119, 'iscrowd': 0, 'isthing': 0, 'area': 3417}, {'id': 3504233, 'category_id': 125, 'iscrowd': 0, 'isthing': 0, 'area': 97238}], 'relations': [[0, 1, 2], [0, 6, 14], [1, 2, 21], [1, 6, 14], [4, 3, 1], [5, 3, 0]], 'annotations': [{'bbox': [49.0, 162.0, 207.0, 617.0], 'bbox_mode': 0, 'category_id': 0}, {'bbox': [259.0,

In [25]:
generated_data = "/home/maelic/Documents/OpenVocSGG/all_annotations/psg_full.json"

with open(generated_data, 'r') as f:
    data = json.load(f)

print(data[0].keys())

for k in data[0].keys():
    print(k, data[0][k])

dict_keys(['image_id', 'dataset', 'data_split', 'objects', 'relationships'])
image_id 417720
dataset psg_full
data_split train2017
objects [{'object_id': 1, 'names': 'person', 'x': 49.0, 'y': 162.0, 'w': 158.0, 'h': 455.0}, {'object_id': 2, 'names': 'person', 'x': 259.0, 'y': 158.0, 'w': 138.0, 'h': 473.0}, {'object_id': 3, 'names': 'umbrella', 'x': 157.0, 'y': 75.0, 'w': 312.0, 'h': 204.0}, {'object_id': 4, 'names': 'tree', 'x': 0.0, 'y': 0.0, 'w': 480.0, 'h': 131.0}, {'object_id': 5, 'names': 'fence', 'x': 0.0, 'y': 112.0, 'w': 480.0, 'h': 211.0}, {'object_id': 6, 'names': 'sky', 'x': 215.0, 'y': 0.0, 'w': 265.0, 'h': 54.0}, {'object_id': 7, 'names': 'grass', 'x': 0.0, 'y': 285.0, 'w': 480.0, 'h': 355.0}]
relationships [{'subject_id': 1, 'object_id': 2, 'predicate': 'beside', 'relationship_id': 0, 'confidence': 1.0}, {'subject_id': 1, 'object_id': 7, 'predicate': 'standing on', 'relationship_id': 0, 'confidence': 1.0}, {'subject_id': 2, 'object_id': 3, 'predicate': 'holding', 'relati

In [22]:
def bbox_to_xywh(bbox):
    x1, y1, x2, y2 = bbox
    return {'x': x1, 'y': y1, 'w': x2 - x1, 'h': y2 - y1}

pred_classes = psg['predicate_classes']
with open('/home/maelic/Documents/SGG-Benchmark/datasets/psg/obj_classes.txt', 'r') as f:
    obj_classes = f.read().splitlines()

new_psg_data = []

obj_id = 0
relationship_id = 0

for d in tqdm(psg['data']):
    obj_idx_to_id = {}
    current_dict = {}
    current_dict['image_id'] = d['coco_image_id']
    current_dict['dataset'] = 'psg_full'
    # if d['image_id'] in psg['test_image_ids']:
    #     current_dict['data_split'] = 'test'
    # else:
    current_dict['data_split'] = d['file_name'].split('/')[0]
    current_dict['objects'] = []
    current_dict['relationships'] = []

    for i, o in enumerate(d['annotations']):
        obj_id += 1
        cat = obj_classes[o['category_id']]
        box = bbox_to_xywh(o['bbox'])
        cur_obj = {
            'object_id': obj_id,
            'names': cat,
        }
        obj_idx_to_id[i] = obj_id
        cur_obj.update(box)
        current_dict['objects'].append(cur_obj)

    for r in d['relations']:
        cur_rel = {
            'subject_id': obj_idx_to_id[r[0]],
            'object_id': obj_idx_to_id[r[1]],
            'predicate': pred_classes[r[2]],
            'relationship_id': relationship_id,
            'confidence': 1.0,
        }
        relationship_id += 1

        current_dict['relationships'].append(cur_rel)
    new_psg_data.append(current_dict)

print(len(new_psg_data))

# to save the new data
with open('/home/maelic/Documents/OpenVocSGG/psg_full.json', 'w') as f:
    json.dump(new_psg_data, f)

100%|██████████| 46697/46697 [00:00<00:00, 50225.01it/s]


46697


In [23]:
from collections import Counter

split_counter = Counter()
for d in new_psg_data:
    split_counter[d['data_split']] += 1

print(split_counter)

Counter({'train2017': 45697, 'val2017': 1000})


In [1]:
vg150_data = "/home/maelic/Documents/SGG-Benchmark/datasets/VG150/VG-SGG-with-attri.h5"

import h5py
import numpy as np
import json
from tqdm import tqdm

data = h5py.File(vg150_data, 'r')
print(data.keys())

classes = "/home/maelic/Documents/SGG-Benchmark/datasets/VG150/VG-SGG-dicts-with-attri.json"

with open(classes, 'r') as f:
    classes = json.load(f)

image_data = "/home/maelic/Documents/SGG-Benchmark/datasets/VG150/image_data.json"

with open(image_data, 'r') as f:
    image_data = json.load(f)

<KeysViewHDF5 ['active_object_mask', 'attributes', 'boxes_1024', 'boxes_512', 'img_to_first_box', 'img_to_first_rel', 'img_to_last_box', 'img_to_last_rel', 'labels', 'predicates', 'relationships', 'split']>


In [2]:
obj_classes = classes['idx_to_label']
pred_classes = classes['idx_to_predicate']

print(image_data[0])

for k, v in data.items():
    print(k, v[1000])

{'width': 800, 'url': 'https://cs.stanford.edu/people/rak248/VG_100K_2/1.jpg', 'height': 600, 'image_id': 1, 'coco_id': None, 'flickr_id': None, 'anti_prop': 0.0}
active_object_mask [ True]
attributes [1 0 0 0 0 0 0 0 0 0]
boxes_1024 [787 429  69  98]
boxes_512 [393 214  35  49]
img_to_first_box 10687
img_to_first_rel 6110
img_to_last_box 10697
img_to_last_rel 6114
labels [28]
predicates [31]
relationships [1847 1852]
split 0


In [6]:
def bbox_to_xywh(bbox):
    x1, y1, x2, y2 = bbox
    return {'x': x1, 'y': y1, 'w': x2, 'h': y2}

new_vg150_data = []

obj_id = 0
relationship_id = 0

for i in tqdm(range(len(data['img_to_first_box']))):
    if (data['img_to_first_box'][i] == -1) or (data['img_to_first_rel'][i] == -1) or (data['img_to_first_rel'][i] == data['img_to_last_rel'][i]+1):
        continue
    obj_idx_to_id = {}
    current_dict = {}
    current_dict['image_id'] = image_data[i]['image_id']
    current_dict['dataset'] = 'vg150_full'
    img_width, img_height = image_data[i]['width'], image_data[i]['height']
    # if d['image_id'] in psg['test_image_ids']:
    #     current_dict['data_split'] = 'test'
    # else:
    if data['split'][i] == 0:
        current_dict['data_split'] = 'train'
    elif data['split'][i] == 1:
        current_dict['data_split'] = 'val'
    else:
        current_dict['data_split'] = 'test'
    current_dict['objects'] = []
    current_dict['relationships'] = []

    for j in range(data['img_to_first_box'][i], data['img_to_last_box'][i]+1):
        obj_id += 1
        cat = obj_classes[str(data['labels'][j][0])]
        box_1024 = data['boxes_1024'][j] / 1024 * max(img_width, img_height)
        # from x_center, y_center to x1, y1, x2, y2
        box_1024[0] -= box_1024[2] / 2
        box_1024[1] -= box_1024[3] / 2

        # rescale to original size
        box = bbox_to_xywh(box_1024)
        cur_obj = {
            'object_id': obj_id,
            'names': cat,
        }
        obj_idx_to_id[j] = obj_id
        cur_obj.update(box)
        current_dict['objects'].append(cur_obj)

    for k in range(data['img_to_first_rel'][i], data['img_to_last_rel'][i]+1):
        r = data['relationships'][k]
        pred = data['predicates'][k][0]
        to_pass = False
        for p in current_dict['relationships']:
            if p['subject_id'] == obj_idx_to_id[r[0]] and p['object_id'] == obj_idx_to_id[r[1]]:
                to_pass =True
                break
        if to_pass:
            continue
        cur_rel = {
            'subject_id': obj_idx_to_id[r[0]],
            'object_id': obj_idx_to_id[r[1]],
            'predicate': pred_classes[str(pred)],
            'relationship_id': relationship_id,
            'confidence': 1.0,
        }
        relationship_id += 1

        current_dict['relationships'].append(cur_rel)
    new_vg150_data.append(current_dict)

print(len(new_vg150_data))

# to save the new data
with open('/home/maelic/Documents/OpenVocSGG/vg150_full2.json', 'w') as f:
    json.dump(new_vg150_data, f)

100%|██████████| 108073/108073 [09:13<00:00, 195.13it/s]


89169


In [33]:
from collections import Counter

split_counter = Counter()
for d in new_vg150_data:
    split_counter[d['data_split']] += 1

print(split_counter)

Counter({'train': 75651, 'test': 32422})
